In [2]:
import logging
import requests
import pandas as pd
from google.cloud import bigquery

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
print('Libraries imported successfully')

Libraries imported successfully


In [3]:
API_URL = 'https://dummyjson.com/products'

response = requests.get(API_URL, params={'limit': 0}, timeout=30)
response.raise_for_status()

payload = response.json()
products = payload['products']

print('API total:', payload.get('total'))
print('Products fetched:', len(products))
print('\nFirst raw product:')
print(products[0])

API total: 194
Products fetched: 194

First raw product:
{'id': 1, 'title': 'Essence Mascara Lash Princess', 'description': 'The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.', 'category': 'beauty', 'price': 9.99, 'discountPercentage': 10.48, 'rating': 2.56, 'stock': 99, 'tags': ['beauty', 'mascara'], 'brand': 'Essence', 'sku': 'BEA-ESS-ESS-001', 'weight': 4, 'dimensions': {'width': 15.14, 'height': 13.08, 'depth': 22.99}, 'warrantyInformation': '1 week warranty', 'shippingInformation': 'Ships in 3-5 business days', 'availabilityStatus': 'In Stock', 'reviews': [{'rating': 3, 'comment': 'Would not recommend!', 'date': '2025-04-30T09:41:02.053Z', 'reviewerName': 'Eleanor Collins', 'reviewerEmail': 'eleanor.collins@x.dummyjson.com'}, {'rating': 4, 'comment': 'Very satisfied!', 'date': '2025-04-30T09:41:02.053Z', 'reviewerName': 'Lucas Gordon', 'reviewerEmail': 'l

In [4]:
df_raw = pd.json_normalize(products)

print('Raw DataFrame shape:', df_raw.shape)
display(df_raw.head(3))

Raw DataFrame shape: (194, 27)


,id,title,description,category,price,discountPercentage,rating,stock,tags,brand,...,minimumOrderQuantity,images,thumbnail,dimensions.width,dimensions.height,dimensions.depth,meta.createdAt,meta.updatedAt,meta.barcode,meta.qrCode
0,1,Essence Mascara Lash Princess,The Essence Mascara Lash Princess is a popular...,beauty,9.99,10.48,2.56,99,"[beauty, mascara]",Essence,...,48,[https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...,15.14,13.08,22.99,2025-10-09T14:47:01.588Z,2026-05-23T11:27:41.868Z,5784719087687,https://cdn.dummyjson.com/public/qr-code.png
1,2,Eyeshadow Palette with Mirror,The Eyeshadow Palette with Mirror offers a ver...,beauty,19.99,18.19,2.86,34,"[beauty, eyeshadow]",Glamour Beauty,...,20,[https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...,9.26,22.47,27.67,2024-12-03T11:29:45.706Z,2025-08-24T16:52:47.806Z,9170275171413,https://cdn.dummyjson.com/public/qr-code.png
2,3,Powder Canister,The Powder Canister is a finely milled setting...,beauty,14.99,9.84,4.64,89,"[beauty, face powder]",Velvet Touch,...,22,[https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...,29.27,27.93,20.59,2025-01-08T18:30:12.844Z,2025-11-26T14:49:26.980Z,8418883906837,https://cdn.dummyjson.com/public/qr-code.png


In [5]:
columns = [
    'id', 'title', 'category', 'price', 'discountPercentage',
    'rating', 'stock', 'brand', 'availabilityStatus'
]

df = df_raw[columns].copy()
df = df.rename(columns={
    'id': 'product_id',
    'title': 'product_name',
    'discountPercentage': 'discount_percentage',
    'availabilityStatus': 'availability_status'
})

display(df.head())

,product_id,product_name,category,price,discount_percentage,rating,stock,brand,availability_status
0,1,Essence Mascara Lash Princess,beauty,9.99,10.48,2.56,99,Essence,In Stock
1,2,Eyeshadow Palette with Mirror,beauty,19.99,18.19,2.86,34,Glamour Beauty,In Stock
2,3,Powder Canister,beauty,14.99,9.84,4.64,89,Velvet Touch,In Stock
3,4,Red Lipstick,beauty,12.99,12.16,4.36,91,Chic Cosmetics,In Stock
4,5,Red Nail Polish,beauty,8.99,11.44,4.32,79,Nail Couture,In Stock


In [6]:
text_columns = ['product_name', 'category', 'brand', 'availability_status']
for column in text_columns:
    df[column] = df[column].astype('string').str.strip()

df['category'] = df['category'].str.lower()

numeric_columns = ['product_id', 'price', 'discount_percentage', 'rating', 'stock']
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

print('Types after conversion:')
print(df.dtypes)

Types after conversion:
product_id                      int64
product_name           string[python]
category               string[python]
price                         float64
discount_percentage           float64
rating                        float64
stock                           int64
brand                  string[python]
availability_status    string[python]
dtype: object


In [7]:
print('Missing values before handling:')
display(df.isna().sum().to_frame('missing_count'))

print('Duplicate product IDs:', df['product_id'].duplicated().sum())
print('Rows before cleaning:', len(df))

Missing values before handling:


,missing_count
product_id,0
product_name,0
category,0
price,0
discount_percentage,0
rating,0
stock,0
brand,92
availability_status,0


Duplicate product IDs: 0
Rows before cleaning: 194


In [8]:
# Remove rows missing required fields or containing invalid core values
df = df.dropna(subset=['product_id', 'product_name', 'price', 'stock'])
df = df[df['product_name'].ne('') & df['price'].ge(0) & df['stock'].ge(0)]

# Remove duplicate products by ID
df = df.drop_duplicates(subset=['product_id'], keep='last')

# Validate optional values; invalid values become missing
df.loc[~df['rating'].between(0, 5), 'rating'] = float('nan')
df.loc[~df['discount_percentage'].between(0, 100), 'discount_percentage'] = float('nan')

# Keep products with no listed brand, but make the missing value explicit
df['brand'] = df['brand'].fillna('Unknown')
df['brand'] = df['brand'].replace('', 'Unknown')

df['product_id'] = df['product_id'].astype('int64')
df['stock'] = df['stock'].astype('int64')

df = df.reset_index(drop=True)

print('Rows after cleaning:', len(df))
print('Duplicate product IDs after cleaning:', df['product_id'].duplicated().sum())
print('Missing values after cleaning:')
display(df.isna().sum().to_frame('missing_count'))
display(df.head())

Rows after cleaning: 194
Duplicate product IDs after cleaning: 0
Missing values after cleaning:


,missing_count
product_id,0
product_name,0
category,0
price,0
discount_percentage,0
rating,0
stock,0
brand,0
availability_status,0


,product_id,product_name,category,price,discount_percentage,rating,stock,brand,availability_status
0,1,Essence Mascara Lash Princess,beauty,9.99,10.48,2.56,99,Essence,In Stock
1,2,Eyeshadow Palette with Mirror,beauty,19.99,18.19,2.86,34,Glamour Beauty,In Stock
2,3,Powder Canister,beauty,14.99,9.84,4.64,89,Velvet Touch,In Stock
3,4,Red Lipstick,beauty,12.99,12.16,4.36,91,Chic Cosmetics,In Stock
4,5,Red Nail Polish,beauty,8.99,11.44,4.32,79,Nail Couture,In Stock


In [9]:
df.to_csv('cleaned_products.csv', index=False)
print('Saved cleaned_products.csv')

Saved cleaned_products.csv


In [10]:
PROJECT_ID = 'promising-node-513006-f9'
DATASET_ID = 'ecommerce'
TABLE_ID = 'products'
TABLE_PATH = f'{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}'

print('Target table:', TABLE_PATH)

Target table: promising-node-513006-f9.ecommerce.products


In [11]:
client = bigquery.Client(project=PROJECT_ID)

job_config = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField('product_id', 'INT64', mode='REQUIRED'),
        bigquery.SchemaField('product_name', 'STRING'),
        bigquery.SchemaField('category', 'STRING'),
        bigquery.SchemaField('price', 'FLOAT64'),
        bigquery.SchemaField('discount_percentage', 'FLOAT64'),
        bigquery.SchemaField('rating', 'FLOAT64'),
        bigquery.SchemaField('stock', 'INT64'),
        bigquery.SchemaField('brand', 'STRING'),
        bigquery.SchemaField('availability_status', 'STRING'),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

load_job = client.load_table_from_dataframe(
    df,
    TABLE_PATH,
    job_config=job_config,
)
load_job.result()

table = client.get_table(TABLE_PATH)
print(f'Loaded {table.num_rows} rows into {TABLE_PATH}')

DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.